## GeoLocation data EDA

In [6]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import folium

In [7]:
geolocation = pd.read_csv("../../data/raw/olist_geolocation_dataset.csv")
geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [8]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [9]:
geolocation.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [10]:
m = folium.Map(
    location=[-14.2, -51.9],
    zoom_start=4
)

In [11]:
# m removing maps becasue file getting too large

In [12]:
from folium.plugins import HeatMap

In [13]:
heat_data = geolocation[
    ["geolocation_lat", "geolocation_lng"]
].values.tolist()

HeatMap(
    heat_data,
    radius=10,
    blur=15,
    max_zoom=6
).add_to(m)



In [14]:
# m -----------------removing maps becasue file getting too large

In [15]:
# m.save("../../output/maps/geolocation_heatmap.html") use this to save html in your output folder

In [16]:
geolocation["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(981148)

In [17]:
geolocation["geolocation_zip_code_prefix"].nunique()

19015

In [18]:
geolocation.groupby(
    "geolocation_zip_code_prefix"
).size().describe()

count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64

In [19]:
geo_by_zip = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean"
    })
    .reset_index()
)

In [20]:
geo_by_zip.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


In [21]:
geo_by_zip["geolocation_zip_code_prefix"].nunique()

19015

In [22]:
customer_geo = customers.merge(
    geo_by_zip,
    how="left",
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix"
)

In [23]:
customer_geo

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409.0,-20.498489,-47.396929
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,9790.0,-23.727992,-46.542848
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1151.0,-23.531642,-46.656289
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,8775.0,-23.499702,-46.185233
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,13056.0,-22.975100,-47.142925
...,...,...,...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP,3937.0,-23.586003,-46.499638
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP,6764.0,-23.615830,-46.768533
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE,60115.0,-3.734569,-38.510534
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS,92120.0,-29.949839,-51.168494


In [24]:

customer_geo.isnull().sum()


customer_id                      0
customer_unique_id               0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
geolocation_zip_code_prefix    278
geolocation_lat                278
geolocation_lng                278
dtype: int64

In [25]:
customer_geo=customer_geo.dropna()

In [26]:
customer_heatmap = folium.Map(
    location=[-14.2, -51.9],
    zoom_start=4
)
heat_data = customer_geo[
    ["geolocation_lat", "geolocation_lng"]
].values.tolist()

HeatMap(
    heat_data,
    radius=10,
    blur=15,
    max_zoom=6
).add_to(customer_heatmap)

In [27]:
# customer_heatmap  #use this to render graph 

In [28]:
# customer_heatmap.save("../../output/maps/customer_location_heatmap.html")